In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any

import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont
from torch import Tensor

# Config

In [ ]:
cfg_mobilenet_v2 = {
    "name": "mobilenet_v2",
    # anchor
    "min_sizes": [[16, 32], [64, 128], [256, 512]],
    "steps": [8, 16, 32],
    "variance": [0.1, 0.2],
    "clip": False,
    # loss
    "loc_weight": 2.0,
    # training
    "batch_size": 32,
    "epochs": 250,
    "milestones": [190, 220],
    # input
    "image_size": 640,
    # backbone
    "pretrain": True,
    "return_layers": [6, 13, 18],
    "in_channel": 32,
    "out_channel": 128,
}

In [ ]:
from copy import deepcopy

_CONFIGS = {
    "mobilenet_v2": cfg_mobilenet_v2,
}


def get_config(name: str):
    """
    Get configuration by model name.

    Args:
        name (str): Name of the backbone/model.

    Returns:
        dict: A deep copy of the configuration.

    Raises:
        ValueError: If config name is not found.
    """
    if name not in _CONFIGS:
        available = ", ".join(_CONFIGS.keys())
        raise ValueError(f"Unknown config '{name}'. Available: [{available}]")

    return deepcopy(_CONFIGS[name])

# Box Utils

In [ ]:
def decode(
    loc: Tensor, priors: Tensor, variances: list[float] | tuple[float, float]
) -> Tensor:
    """Decode predicted box deltas back to point-form boxes."""
    boxes = torch.cat(
        (
            priors[:, :2] + loc[:, :2] * variances[0] * priors[:, 2:],
            priors[:, 2:] * torch.exp(loc[:, 2:] * variances[1]),
        ),
        1,
    )
    boxes[:, :2] -= boxes[:, 2:] / 2
    boxes[:, 2:] += boxes[:, :2]
    return boxes


def decode_landm(
    pre: Tensor, priors: Tensor, variances: list[float] | tuple[float, float]
) -> Tensor:
    """Decode predicted landmark deltas back to normalized coordinates."""
    decoded = pre.view(-1, 5, 2) * variances[0] * priors[:, 2:].unsqueeze(1)
    decoded += priors[:, :2].unsqueeze(1)
    return decoded.reshape(-1, 10)

# Prior Box

In [ ]:
from __future__ import annotations

from math import ceil

import torch
from torch import Tensor


class PriorBox:
    """
    Generate RetinaFace priors in ``(cx, cy, w, h)`` format.

    The generated coordinates are normalized to the input image size so they can
    be consumed directly by the box encode/decode logic.
    """

    def __init__(self, cfg: dict, image_size: tuple[int, int]) -> None:
        self.image_size = image_size

        self.min_sizes: list[list[int]] = cfg["min_sizes"]
        self.steps: list[int] = cfg["steps"]
        self.clip: bool = cfg["clip"]

        if len(self.min_sizes) != len(self.steps):
            raise ValueError(
                "cfg['min_sizes'] and cfg['steps'] must have the same length"
            )

        image_height, image_width = self.image_size
        self.feature_maps: list[list[int]] = [
            [ceil(image_height / step), ceil(image_width / step)] for step in self.steps
        ]

    def forward(self) -> Tensor:
        anchors: list[float] = []
        image_height, image_width = self.image_size

        for level_idx, feature_map in enumerate(self.feature_maps):
            min_sizes = self.min_sizes[level_idx]
            step = self.steps[level_idx]

            for row in range(feature_map[0]):
                for col in range(feature_map[1]):
                    cx = (col + 0.5) * step / image_width
                    cy = (row + 0.5) * step / image_height

                    for min_size in min_sizes:
                        scale_x = min_size / image_width
                        scale_y = min_size / image_height
                        anchors.extend([cx, cy, scale_x, scale_y])

        priors = torch.tensor(anchors, dtype=torch.float32).view(-1, 4)
        if self.clip:
            priors.clamp_(min=0.0, max=1.0)
        return priors

# Retina Model

## Utils

In [ ]:
from collections import OrderedDict

from torch import nn, Tensor


def _make_divisible(v: float, divisor: int = 8) -> int:
    """This function ensures that all layers have a channel number divisible by 8"""
    new_v = max(divisor, int(v + divisor / 2) // divisor * divisor)
    # Make sure that round down does not go down by more than 10%.
    if new_v < 0.9 * v:
        new_v += divisor
    return new_v


class IntermediateLayerGetterByIndex(nn.Module):
    def __init__(self, model: nn.Module, indexes: list[int] | None = None) -> None:
        super().__init__()
        features = getattr(model, "features", None)
        if not isinstance(features, nn.Sequential):
            raise TypeError("model.features must be an nn.Sequential")
        self.features: nn.Sequential = features
        self.indexes = tuple(indexes or [6, 13, 18])

    def forward(self, x: Tensor) -> OrderedDict[str, Tensor]:
        outputs: OrderedDict[str, Tensor] = OrderedDict()
        for idx, layer in enumerate(self.features):
            x = layer(x)
            if idx in self.indexes:
                out_name = f"layer_{idx}"
                outputs[out_name] = x

        return outputs

## Backbone

In [ ]:
from typing import Any

import torch
from torch import nn, Tensor
from torchvision.models import MobileNet_V2_Weights
from typing import Any, Callable


class Conv2dNormActivation(nn.Sequential):
    """Convolutional block, consists of nn.Conv2d, nn.BatchNorm2d, nn.ReLU"""

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int | None = None,
        groups: int = 1,
        norm_layer: Callable[..., torch.nn.Module] | None = torch.nn.BatchNorm2d,
        activation_layer: Callable[..., torch.nn.Module] | None = torch.nn.LeakyReLU,
        dilation: int = 1,
        inplace: bool | None = True,
        negative_slope: float | None = None,
        bias: bool = False,
    ) -> None:
        if padding is None:
            padding = (kernel_size - 1) // 2 * dilation

        layers: list[nn.Module] = [
            nn.Conv2d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=kernel_size,
                stride=stride,
                padding=padding,
                dilation=dilation,
                groups=groups,
                bias=bias,
            )
        ]
        if norm_layer is not None:
            layers.append(norm_layer(out_channels))

        if activation_layer is not None:
            params: dict[str, Any] = {} if inplace is None else {"inplace": inplace}
            if negative_slope is not None:
                params["negative_slope"] = negative_slope
            layers.append(activation_layer(**params))
        super().__init__(*layers)
        self.in_channels = in_channels
        self.out_channels = out_channels


__all__ = ["mobilenet_v2"]


class InvertedResidual(nn.Module):
    def __init__(
        self, in_planes: int, out_planes: int, stride: int, expand_ratio: int
    ) -> None:
        super().__init__()
        self.stride = stride
        if stride not in [1, 2]:
            raise ValueError(f"stride should be 1 or 2 instead of {stride}")

        hidden_dim = int(round(in_planes * expand_ratio))
        self.use_res_connect = self.stride == 1 and in_planes == out_planes

        layers: list[nn.Module] = []
        if expand_ratio != 1:
            # pw
            layers.append(
                Conv2dNormActivation(
                    in_planes, hidden_dim, kernel_size=1, activation_layer=nn.ReLU6
                )
            )
        layers.extend(
            [
                # dw
                Conv2dNormActivation(
                    hidden_dim,
                    hidden_dim,
                    stride=stride,
                    groups=hidden_dim,
                    activation_layer=nn.ReLU6,
                ),
                # pw-linear
                nn.Conv2d(hidden_dim, out_planes, 1, 1, 0, bias=False),
                nn.BatchNorm2d(out_planes),
            ]
        )
        self.conv = nn.Sequential(*layers)
        self.out_channels = out_planes
        self._is_cn = stride > 1

    def forward(self, x: Tensor) -> Tensor:
        if self.use_res_connect:
            return x + self.conv(x)
        else:
            return self.conv(x)


class MobileNetV2(nn.Module):
    def __init__(
        self,
        num_classes: int = 1000,
        width_mult: float = 1.0,
        inverted_residual_setting: list[list[int]] | None = None,
        round_nearest: int = 8,
        dropout: float = 0.2,
    ) -> None:
        """
        MobileNet V2 main class

        Args:
            num_classes (int): Number of classes
            width_mult (float): Width multiplier - adjusts number of channels in each layer by this amount
            inverted_residual_setting: Network structure
            round_nearest (int): Round the number of channels in each layer to be a multiple of this number
            Set to 1 to turn off rounding
            block: Module specifying inverted residual building block for mobilenet
            dropout (float): The droupout probability

        """
        super().__init__()

        input_channel = 32
        last_channel = 1280

        if inverted_residual_setting is None:
            inverted_residual_setting = [
                # t, c, n, s
                [1, 16, 1, 1],
                [6, 24, 2, 2],
                [6, 32, 3, 2],
                [6, 64, 4, 2],
                [6, 96, 3, 1],
                [6, 160, 3, 2],
                [6, 320, 1, 1],
            ]

        # only check the first element, assuming user knows t,c,n,s are required
        if (
            len(inverted_residual_setting) == 0
            or len(inverted_residual_setting[0]) != 4
        ):
            raise ValueError(
                f"inverted_residual_setting should be non-empty or a 4-element list, got {inverted_residual_setting}"
            )

        # building first layer
        input_channel = _make_divisible(input_channel * width_mult, round_nearest)
        self.last_channel = _make_divisible(
            last_channel * max(1.0, width_mult), round_nearest
        )
        features: list[nn.Module] = [
            Conv2dNormActivation(3, input_channel, stride=2, activation_layer=nn.ReLU6)
        ]
        # building inverted residual blocks
        for t, c, n, s in inverted_residual_setting:
            output_channel = _make_divisible(c * width_mult, round_nearest)
            for i in range(n):
                stride = s if i == 0 else 1
                features.append(
                    InvertedResidual(
                        input_channel, output_channel, stride, expand_ratio=t
                    )
                )
                input_channel = output_channel
        # building last several layers
        features.append(
            Conv2dNormActivation(
                input_channel,
                self.last_channel,
                kernel_size=1,
                activation_layer=nn.ReLU6,
            )
        )
        # make it nn.Sequential
        self.features = nn.Sequential(*features)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        # building classifier
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(self.last_channel, num_classes),
        )

        # weight initialization
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.zeros_(m.bias)

    def forward(self, x: Tensor) -> Tensor:
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)

        x = self.classifier(x)
        return x


def mobilenet_v2(
    *, pretrained: bool = True, progress: bool = True, **kwargs: Any
) -> MobileNetV2:

    if pretrained:
        weights = MobileNet_V2_Weights.IMAGENET1K_V2
    else:
        weights = None

    model = MobileNetV2(**kwargs)

    if weights is not None:
        state_dict = weights.get_state_dict(progress=progress, check_hash=True)
        model.load_state_dict(state_dict)

    return model


## Builder

In [ ]:
BACKBONE_FACTORY = {
    "mobilenet_v2": mobilenet_v2,
}


def build_backbone(name, pretrained=False):
    if name not in BACKBONE_FACTORY:
        raise ValueError(f"Unknown backbone: {name}")

    return BACKBONE_FACTORY[name](pretrained=pretrained)


def get_layer_extractor(config, backbone):
    if config["name"] == "mobilenet_v2":
        return IntermediateLayerGetterByIndex(backbone, indexes=config["return_layers"])

    raise ValueError(f"Unsupported layer extractor for backbone: {config['name']}")

## Heads

In [ ]:
import torch
from torch import Tensor, nn


class BboxHead(nn.Module):
    def __init__(
        self, in_channels: int = 512, num_anchors: int = 2, fpn_num: int = 3
    ) -> None:
        super().__init__()
        self.bbox_head = nn.ModuleList(
            [
                nn.Conv2d(
                    in_channels=in_channels,
                    out_channels=num_anchors * 4,
                    kernel_size=(1, 1),
                    stride=1,
                    padding=0,
                )
                for _ in range(fpn_num)
            ]
        )

    def forward(self, x: list[Tensor]) -> Tensor:
        output_tensors: list[Tensor] = []
        for feature, layer in zip(x, self.bbox_head):
            output_tensors.append(layer(feature).permute(0, 2, 3, 1).contiguous())

        bbox_offsets = torch.cat(
            [out.view(out.shape[0], -1, 4) for out in output_tensors], dim=1
        )
        return bbox_offsets

In [ ]:
import torch
from torch import Tensor, nn


class ClassHead(nn.Module):
    def __init__(
        self, in_channels: int = 512, num_anchors: int = 2, fpn_num: int = 3
    ) -> None:
        super().__init__()
        self.class_head = nn.ModuleList(
            [
                nn.Conv2d(
                    in_channels=in_channels,
                    out_channels=num_anchors * 2,
                    kernel_size=(1, 1),
                    stride=1,
                    padding=0,
                )
                for _ in range(fpn_num)
            ]
        )

    def forward(self, x: list[Tensor]) -> Tensor:
        output_tensors: list[Tensor] = []
        for feature, layer in zip(x, self.class_head):
            output_tensors.append(layer(feature).permute(0, 2, 3, 1).contiguous())

        class_scores = torch.cat(
            [out.view(out.shape[0], -1, 2) for out in output_tensors], dim=1
        )
        return class_scores

In [ ]:
import torch
from torch import Tensor, nn


class LandmarkHead(nn.Module):
    def __init__(
        self, in_channels: int = 512, num_anchors: int = 2, fpn_num: int = 3
    ) -> None:
        super().__init__()
        self.landmark_head = nn.ModuleList(
            [
                nn.Conv2d(
                    in_channels,
                    num_anchors * 10,
                    kernel_size=(1, 1),
                    stride=1,
                    padding=0,
                )
                for _ in range(fpn_num)
            ]
        )

    def forward(self, x: list[Tensor]) -> Tensor:
        output_tensors: list[Tensor] = []
        for feature, layer in zip(x, self.landmark_head):
            output_tensors.append(layer(feature).permute(0, 2, 3, 1).contiguous())

        landmarks = torch.cat(
            [out.view(out.shape[0], -1, 10) for out in output_tensors], dim=1
        )
        return landmarks

## Necks

In [ ]:
from __future__ import annotations

import torch.nn.functional as F
from torch import Tensor, nn



class FPN(nn.Module):
    """
    FPN (Feature Pyramid Network) for multi-scale feature map extraction and merging.
    Uses 1x1 convolutions for output layers and 3x3 convolutions for merging layers.
    """

    def __init__(self, in_channels_list: list[int], out_channels: int) -> None:
        """
        Initializes the FPN module.

        Args:
            in_channels_list (list of int): List of input channel sizes for each pyramid level.
            out_channels (int): Number of output channels for the feature pyramid.
        """
        super().__init__()
        leaky = 0.1 if out_channels <= 64 else 0
        # Define 1x1 convolution output layers
        self.output1 = Conv2dNormActivation(
            in_channels_list[0], out_channels, kernel_size=1, negative_slope=leaky
        )
        self.output2 = Conv2dNormActivation(
            in_channels_list[1], out_channels, kernel_size=1, negative_slope=leaky
        )
        self.output3 = Conv2dNormActivation(
            in_channels_list[2], out_channels, kernel_size=1, negative_slope=leaky
        )

        # Define merge layers using 3x3 convolutions
        self.merge1 = Conv2dNormActivation(
            out_channels, out_channels, kernel_size=3, negative_slope=leaky
        )
        self.merge2 = Conv2dNormActivation(
            out_channels, out_channels, kernel_size=3, negative_slope=leaky
        )

    def forward(self, inputs) -> list[Tensor]:
        """
        Forward pass of the FPN module.

        Args:
            inputs (dict or list): Input feature maps from different levels of the pyramid.

        Returns:
            list: List of merged output feature maps at different scales.
        """
        inputs = list(inputs.values())

        # Apply output layers to each feature map
        output1 = self.output1(inputs[0])
        output2 = self.output2(inputs[1])
        output3 = self.output3(inputs[2])

        # Merge outputs with upsampling and addition
        upsample3 = F.interpolate(output3, size=output2.shape[2:], mode="nearest")
        output2 = self.merge2(output2 + upsample3)

        upsample2 = F.interpolate(output2, size=output1.shape[2:], mode="nearest")
        output1 = self.merge1(output1 + upsample2)

        # Return merged feature maps
        return [output1, output2, output3]

In [ ]:
from __future__ import annotations
import torch
import torch.nn.functional as F
from torch import Tensor, nn

__all__ = ["SSH"]


class SSH(nn.Module):
    """
    SSH (Single Stage Headless) Module for feature extraction.
    Combines 3x3, 5x5, and 7x7 convolutions with batch normalization and optional LeakyReLU activations.
    """

    def __init__(self, in_channel: int, out_channels: int) -> None:
        """
        Initializes the SSH module.

        Args:
            in_channel (int): Number of input channels.
            out_channels (int): Number of output channels, must be divisible by 4.
        """
        super().__init__()

        if out_channels % 4 != 0:
            raise ValueError("out_channels must be divisible by 4.")
        leaky = 0.1 if out_channels <= 64 else 0

        # 3x3 Convolution branch
        self.conv3X3 = Conv2dNormActivation(
            in_channel, out_channels // 2, kernel_size=3, activation_layer=None
        )

        # 5x5 Convolution branch
        self.conv5X5_1 = Conv2dNormActivation(
            in_channel, out_channels // 4, kernel_size=3, negative_slope=leaky
        )
        self.conv5X5_2 = Conv2dNormActivation(
            out_channels // 4, out_channels // 4, kernel_size=3, activation_layer=None
        )

        # 7x7 Convolution branch
        self.conv7X7_2 = Conv2dNormActivation(
            out_channels // 4, out_channels // 4, kernel_size=3, negative_slope=leaky
        )
        self.conv7x7_3 = Conv2dNormActivation(
            out_channels // 4, out_channels // 4, kernel_size=3, activation_layer=None
        )

    def forward(self, x: Tensor) -> Tensor:
        """
        Forward pass of the SSH module.

        Args:
            x (Tensor): Input feature map.

        Returns:
            Tensor: Output feature map after applying SSH operations and ReLU activation.
        """
        conv3X3 = self.conv3X3(x)
        conv5X5_1 = self.conv5X5_1(x)
        conv5X5 = self.conv5X5_2(conv5X5_1)
        conv7X7 = self.conv7x7_3(self.conv7X7_2(conv5X5_1))

        out = torch.cat([conv3X3, conv5X5, conv7X7], dim=1)
        return F.relu(out, inplace=True)

## RetinaFace

In [ ]:
from __future__ import annotations

import torch.nn.functional as F
from torch import Tensor, nn

def _resolve_fpn_in_channels(backbone: nn.Module, indexes: list[int]) -> list[int]:
    features = getattr(backbone, "features", None)
    if not isinstance(features, nn.Sequential):
        raise TypeError("backbone.features must be an nn.Sequential")

    channels: list[int] = []
    for idx in indexes:
        if idx < 0 or idx >= len(features):
            raise IndexError(
                f"Feature index {idx} is out of range for backbone.features"
            )

        feature_layer = features[idx]
        out_channels = getattr(feature_layer, "out_channels", None)
        if not isinstance(out_channels, int):
            raise TypeError(
                f"Feature layer at index {idx} does not expose an integer out_channels"
            )
        channels.append(out_channels)

    return channels


class RetinaFace(nn.Module):
    def __init__(self, cfg: dict | None = None) -> None:
        """
        RetinaFace model constructor.

        Args:
            cfg (dict): A configuration dictionary containing model parameters.
        """
        super().__init__()

        if cfg is None:
            raise ValueError("cfg must not be None.")

        backbone = build_backbone(cfg["name"], cfg["pretrain"])
        self.fx = get_layer_extractor(cfg, backbone)  # feature extraction

        num_anchors = 2
        out_channels = cfg["out_channel"]
        feature_indexes = cfg["return_layers"]
        fpn_in_channels = _resolve_fpn_in_channels(backbone, feature_indexes)
        self.fpn_in_channels = fpn_in_channels

        self.fpn = FPN(fpn_in_channels, out_channels)
        self.ssh1 = SSH(out_channels, out_channels)
        self.ssh2 = SSH(out_channels, out_channels)
        self.ssh3 = SSH(out_channels, out_channels)

        self.class_head = ClassHead(
            in_channels=cfg["out_channel"], num_anchors=num_anchors, fpn_num=3
        )
        self.bbox_head = BboxHead(
            in_channels=cfg["out_channel"], num_anchors=num_anchors, fpn_num=3
        )
        self.landmark_head = LandmarkHead(
            in_channels=cfg["out_channel"], num_anchors=num_anchors, fpn_num=3
        )

    def forward(self, x: Tensor) -> tuple[Tensor, Tensor, Tensor]:
        out = self.fx(x)
        fpn = self.fpn(out)

        # single-stage headless module
        feature1 = self.ssh1(fpn[0])
        feature2 = self.ssh2(fpn[1])
        feature3 = self.ssh3(fpn[2])

        features = [feature1, feature2, feature3]

        classifications = self.class_head(features)
        bbox_regressions = self.bbox_head(features)
        landmark_regressions = self.landmark_head(features)

        if self.training:
            output = (bbox_regressions, classifications, landmark_regressions)
        else:
            output = (
                bbox_regressions,
                F.softmax(classifications, dim=-1),
                landmark_regressions,
            )
        return output

# RetinaFace Detector

In [ ]:


__all__ = ["FaceDetector"]


class FaceDetector:
    def __init__(
        self,
        checkpoint_path: str | Path | None = None,
        *,
        config_name: str = "mobilenet_v2",
        device: str | torch.device | None = None,
        image_size: int | None = None,
        bgr_mean: tuple[float, float, float] = (104.0, 117.0, 123.0),
        pad_to_square: bool = True,
        conf_threshold: float = 0.6,
        nms_threshold: float = 0.4,
        top_k: int = 5000,
        keep_top_k: int = 750,
    ) -> None:
        self.device = self._resolve_device(device)
        self.cfg = get_config(config_name)
        if image_size is not None:
            self.cfg["image_size"] = int(image_size)

        self.image_size = int(self.cfg["image_size"])
        self.bgr_mean = tuple(float(value) for value in bgr_mean)
        self.pad_to_square = bool(pad_to_square)
        self.conf_threshold = float(conf_threshold)
        self.nms_threshold = float(nms_threshold)
        self.top_k = int(top_k)
        self.keep_top_k = int(keep_top_k)

        # Backbone pretraining is only relevant for training initialization.
        self.cfg["pretrain"] = False
        self.model = RetinaFace(self.cfg).to(self.device)
        self.model.eval()

        self.priors = self._build_priors().to(self.device)

        if checkpoint_path is not None:
            self.load_checkpoint(checkpoint_path)

    def load_checkpoint(
        self,
        checkpoint_path: str | Path,
        *,
        strict: bool = True,
    ) -> dict[str, Any]:
        checkpoint = torch.load(Path(checkpoint_path), map_location=self.device)

        if isinstance(checkpoint, dict):
            if "model" in checkpoint:
                state_dict = checkpoint["model"]
            elif "state_dict" in checkpoint:
                state_dict = checkpoint["state_dict"]
            else:
                state_dict = checkpoint
        else:
            raise TypeError(
                "checkpoint must be a state_dict or a checkpoint dictionary"
            )

        cleaned_state_dict = {
            (key[7:] if key.startswith("module.") else key): value
            for key, value in state_dict.items()
        }
        self.model.load_state_dict(cleaned_state_dict, strict=strict)
        self.model.eval()

        if isinstance(checkpoint, dict) and "config" in checkpoint:
            ckpt_config = checkpoint.get("config")
            if isinstance(ckpt_config, dict) and "image_size" in ckpt_config:
                ckpt_image_size = int(ckpt_config["image_size"])
                if ckpt_image_size != self.image_size:
                    self.image_size = ckpt_image_size
                    self.cfg["image_size"] = ckpt_image_size
                    self.priors = self._build_priors().to(self.device)

        return checkpoint if isinstance(checkpoint, dict) else {}

    @torch.inference_mode()
    def detect(
        self,
        image: str | Path | Image.Image | np.ndarray | Tensor,
        *,
        conf_threshold: float | None = None,
        nms_threshold: float | None = None,
        top_k: int | None = None,
        keep_top_k: int | None = None,
        assume_bgr: bool = False,
    ) -> list[dict[str, Any]]:
        conf_thr = float(
            self.conf_threshold if conf_threshold is None else conf_threshold
        )
        nms_thr = float(self.nms_threshold if nms_threshold is None else nms_threshold)
        pre_nms_top_k = int(self.top_k if top_k is None else top_k)
        post_nms_top_k = int(self.keep_top_k if keep_top_k is None else keep_top_k)

        image_rgb = self._to_numpy_rgb_image(image, assume_bgr=assume_bgr)
        input_tensor, scale = self._preprocess(image_rgb)

        loc, conf, landm = self.model(input_tensor)
        scores = conf.squeeze(0)[:, 1]

        valid = scores > conf_thr
        if valid.sum().item() == 0:
            return []

        boxes = decode(loc.squeeze(0), self.priors, self.cfg["variance"])
        landms = decode_landm(landm.squeeze(0), self.priors, self.cfg["variance"])

        boxes = boxes[valid]
        landms = landms[valid]
        scores = scores[valid]

        scale_box = torch.tensor(
            [
                float(self.image_size) * scale["x"],
                float(self.image_size) * scale["y"],
                float(self.image_size) * scale["x"],
                float(self.image_size) * scale["y"],
            ],
            dtype=boxes.dtype,
            device=boxes.device,
        )
        scale_landm = torch.tensor(
            [
                float(self.image_size) * scale["x"],
                float(self.image_size) * scale["y"],
                float(self.image_size) * scale["x"],
                float(self.image_size) * scale["y"],
                float(self.image_size) * scale["x"],
                float(self.image_size) * scale["y"],
                float(self.image_size) * scale["x"],
                float(self.image_size) * scale["y"],
                float(self.image_size) * scale["x"],
                float(self.image_size) * scale["y"],
            ],
            dtype=landms.dtype,
            device=landms.device,
        )
        boxes = boxes * scale_box
        landms = landms * scale_landm

        original_height = int(scale["original_height"])
        original_width = int(scale["original_width"])
        boxes[:, 0::2] = boxes[:, 0::2].clamp_(0.0, max(float(original_width - 1), 0.0))
        boxes[:, 1::2] = boxes[:, 1::2].clamp_(
            0.0, max(float(original_height - 1), 0.0)
        )
        landms[:, 0::2] = landms[:, 0::2].clamp_(
            0.0, max(float(original_width - 1), 0.0)
        )
        landms[:, 1::2] = landms[:, 1::2].clamp_(
            0.0, max(float(original_height - 1), 0.0)
        )

        order = torch.argsort(scores, descending=True)
        if pre_nms_top_k > 0:
            order = order[:pre_nms_top_k]
        boxes = boxes[order]
        landms = landms[order]
        scores = scores[order]

        keep = self._nms(boxes, scores, threshold=nms_thr)
        if post_nms_top_k > 0:
            keep = keep[:post_nms_top_k]

        boxes = boxes[keep].detach().cpu().numpy()
        landms = landms[keep].detach().cpu().numpy().reshape(-1, 5, 2)
        scores = scores[keep].detach().cpu().numpy()

        detections: list[dict[str, Any]] = []
        for box, score, landmark in zip(boxes, scores, landms):
            detections.append(
                {
                    "bbox": box.tolist(),
                    "score": float(score),
                    "landmarks": landmark.tolist(),
                }
            )
        return detections

    def draw(
        self,
        image: str | Path | Image.Image | np.ndarray | Tensor,
        detections: list[dict[str, Any]],
        *,
        assume_bgr: bool = False,
        box_color: tuple[int, int, int] = (0, 255, 0),
        landmark_color: tuple[int, int, int] = (255, 0, 0),
        text_color: tuple[int, int, int] = (255, 255, 0),
        thickness: int = 2,
        radius: int = 2,
        draw_score: bool = True,
    ) -> np.ndarray:
        image_rgb = self._to_numpy_rgb_image(image, assume_bgr=assume_bgr).copy()
        canvas = Image.fromarray(image_rgb)
        drawer = ImageDraw.Draw(canvas)
        font = ImageFont.load_default()

        for det in detections:
            bbox = np.asarray(det.get("bbox", []), dtype=np.float32)
            if bbox.shape != (4,):
                continue

            x1, y1, x2, y2 = [float(v) for v in bbox]
            for offset in range(max(int(thickness), 1)):
                drawer.rectangle(
                    [(x1 - offset, y1 - offset), (x2 + offset, y2 + offset)],
                    outline=box_color,
                )

            landmarks = np.asarray(det.get("landmarks", []), dtype=np.float32).reshape(
                -1, 2
            )
            for point in landmarks:
                px, py = float(point[0]), float(point[1])
                drawer.ellipse(
                    [(px - radius, py - radius), (px + radius, py + radius)],
                    fill=landmark_color,
                    outline=landmark_color,
                )

            if draw_score and "score" in det:
                score_text = f"{float(det['score']):.3f}"
                text_position = (x1, max(0.0, y1 - 10.0))
                drawer.text(text_position, score_text, fill=text_color, font=font)

        output = np.asarray(canvas, dtype=np.uint8)
        if assume_bgr:
            output = output[:, :, ::-1]
        return output

    def _build_priors(self) -> Tensor:
        return PriorBox(
            self.cfg,
            image_size=(self.image_size, self.image_size),
        ).forward()

    def _preprocess(self, image_rgb: np.ndarray) -> tuple[Tensor, dict[str, float]]:
        original_height, original_width = image_rgb.shape[:2]
        processed = image_rgb

        if self.pad_to_square:
            side = max(original_height, original_width)
            rgb_fill = (
                int(self.bgr_mean[2]),
                int(self.bgr_mean[1]),
                int(self.bgr_mean[0]),
            )
            canvas = np.empty((side, side, 3), dtype=np.uint8)
            canvas[...] = np.asarray(rgb_fill, dtype=np.uint8)
            canvas[:original_height, :original_width] = image_rgb
            processed = canvas

        processed_height, processed_width = processed.shape[:2]
        resampling_module = getattr(Image, "Resampling", Image)
        bilinear = getattr(resampling_module, "BILINEAR")
        resized = np.asarray(
            Image.fromarray(processed, mode="RGB").resize(
                (self.image_size, self.image_size),
                resample=bilinear,
            ),
            dtype=np.uint8,
        )

        image_bgr = resized[:, :, ::-1].astype(np.float32, copy=False)
        image_bgr -= np.asarray(self.bgr_mean, dtype=np.float32)
        tensor = torch.from_numpy(np.ascontiguousarray(image_bgr)).permute(2, 0, 1)
        tensor = tensor.unsqueeze(0).to(self.device)

        scale = {
            "x": float(processed_width) / float(self.image_size),
            "y": float(processed_height) / float(self.image_size),
            "original_width": float(original_width),
            "original_height": float(original_height),
        }
        return tensor, scale

    @staticmethod
    def _resolve_device(device: str | torch.device | None) -> torch.device:
        if device is None:
            return torch.device("cuda" if torch.cuda.is_available() else "cpu")

        resolved = torch.device(device)
        if resolved.type == "cuda" and not torch.cuda.is_available():
            raise RuntimeError("CUDA was requested but is not available")
        return resolved

    @staticmethod
    def _nms(boxes: Tensor, scores: Tensor, threshold: float) -> Tensor:
        if boxes.numel() == 0:
            return torch.empty((0,), dtype=torch.long, device=boxes.device)

        x1 = boxes[:, 0]
        y1 = boxes[:, 1]
        x2 = boxes[:, 2]
        y2 = boxes[:, 3]

        areas = (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)
        order = scores.argsort(descending=True)

        keep: list[int] = []
        while order.numel() > 0:
            i = int(order[0].item())
            keep.append(i)
            if order.numel() == 1:
                break

            rest = order[1:]
            xx1 = torch.maximum(x1[i], x1[rest])
            yy1 = torch.maximum(y1[i], y1[rest])
            xx2 = torch.minimum(x2[i], x2[rest])
            yy2 = torch.minimum(y2[i], y2[rest])

            inter_w = (xx2 - xx1).clamp(min=0)
            inter_h = (yy2 - yy1).clamp(min=0)
            inter = inter_w * inter_h
            union = areas[i] + areas[rest] - inter
            iou = inter / torch.clamp(union, min=1e-12)

            order = rest[iou <= threshold]

        return torch.tensor(keep, dtype=torch.long, device=boxes.device)

    @staticmethod
    def _to_numpy_rgb_image(
        image: str | Path | Image.Image | np.ndarray | Tensor,
        *,
        assume_bgr: bool,
    ) -> np.ndarray:
        if isinstance(image, (str, Path)):
            with Image.open(image) as img:
                return np.asarray(img.convert("RGB"), dtype=np.uint8)

        if isinstance(image, Image.Image):
            return np.asarray(image.convert("RGB"), dtype=np.uint8)

        if isinstance(image, Tensor):
            image = image.detach().cpu()
            if image.dim() != 3:
                raise ValueError("tensor image must have shape (C, H, W) or (H, W, C)")

            if image.shape[0] in {1, 3}:
                array = image.permute(1, 2, 0).numpy()
            else:
                array = image.numpy()

            if array.ndim != 3:
                raise ValueError("tensor image must be 3-dimensional")
            if array.shape[2] == 1:
                array = np.repeat(array, 3, axis=2)

            array = FaceDetector._to_uint8(array)
            if assume_bgr:
                array = array[:, :, ::-1]
            return array

        if isinstance(image, np.ndarray):
            if image.ndim != 3:
                raise ValueError("numpy image must have shape (H, W, C)")
            if image.shape[2] == 1:
                image = np.repeat(image, 3, axis=2)

            array = FaceDetector._to_uint8(image)
            if assume_bgr:
                array = array[:, :, ::-1]
            return array

        raise TypeError(f"unsupported image type: {type(image)!r}")

    @staticmethod
    def _to_uint8(array: np.ndarray) -> np.ndarray:
        if array.dtype == np.uint8:
            return array

        if np.issubdtype(array.dtype, np.floating):
            max_value = float(np.nanmax(array)) if array.size > 0 else 0.0
            if max_value <= 1.0:
                array = array * 255.0

        return np.clip(array, 0, 255).astype(np.uint8)


In [ ]:
checkpoint_path = Path("latest.pth")
image_path = Path("path/to/your_image.jpg")

if not checkpoint_path.exists():
    raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")
if not image_path.exists():
    raise FileNotFoundError(f"Image not found: {image_path}")

detector = FaceDetector(
    checkpoint_path=checkpoint_path,
    conf_threshold=0.6,
    nms_threshold=0.4,
)

detections = detector.detect(image_path)
annotated = detector.draw(
    image_path,
    detections,
    draw_score=True,
    thickness=2,
    radius=2,
)

print(f"Detected {len(detections)} face(s)")
for i, det in enumerate(detections, start=1):
    score = float(det["score"])
    bbox = np.round(np.asarray(det["bbox"], dtype=np.float32), 2).tolist()
    print(f"{i}. score={score:.3f}, bbox={bbox}")

import matplotlib.pyplot as plt

plt.figure(figsize=(12, 8))
plt.imshow(annotated)
plt.title(f"RetinaFace Inference - {len(detections)} face(s)")
plt.axis("off")
plt.show()

output_path = image_path.with_name(f"{image_path.stem}_detected{image_path.suffix}")
Image.fromarray(annotated).save(output_path)
print(f"Saved result to: {output_path}")